# DWS Water Quality Database Scraper
**Goal**: Download ALL surface water quality stations from the South African Department of Water and Sanitation, build a unified database, and match to our EY Datathon test set.

**How it works**:
1. Scrape station lists from each drainage region page (A-X)
2. Download all available ZIP data files  
3. Extract and combine into one database
4. Match to test locations by proximity and date

**Output**: A CSV with matched TAL (Alkalinity), EC (Conductance), and PO4_P (Phosphorus) for each test row.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/EY_Datathon_DWS'
ZIP_DIR = os.path.join(BASE_DIR, 'zips')
CSV_DIR = os.path.join(BASE_DIR, 'csvs')
os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
print(f"Working directory: {BASE_DIR}")
print(f"ZIPs will be saved to: {ZIP_DIR}")
print(f"CSVs will be extracted to: {CSV_DIR}")


Mounted at /content/drive
Working directory: /content/drive/MyDrive/EY_Datathon_DWS
ZIPs will be saved to: /content/drive/MyDrive/EY_Datathon_DWS/zips
CSVs will be extracted to: /content/drive/MyDrive/EY_Datathon_DWS/csvs


In [ ]:
import requests
import pandas as pd
import numpy as np
import re
import zipfile
import io
import time
import glob
from bs4 import BeautifulSoup
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Upload your test file to Drive, or adjust this path
# TEST_FILE = os.path.join(BASE_DIR, 'testing_all.csv')


## Step 1: Scrape Station Lists from All Regions

The DWS site organizes stations by primary drainage region (A-X). Each region page has an HTML table with station ID, description, coordinates, sample count, and a link to download the data ZIP.


In [ ]:
def scrape_region(region_letter):
    """
    Scrape a DWS region page to extract all surface water stations.
    Returns a list of dicts with station info.
    """
    url = f"https://www.dws.gov.za/iwqs/wms/data/{region_letter}_reg_WMS_nobor.htm"

    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"  Region {region_letter}: Failed to fetch ({e})")
        return []

    soup = BeautifulSoup(resp.content, 'html.parser')
    stations = []

    # Find all table rows
    for table in soup.find_all('table'):
        rows = table.find_all('tr')
        for row in rows:
            cells = row.find_all('td')
            if len(cells) < 10:
                continue

            # Try to extract station info from the row
            # The format varies but typically:
            # col 0: station name (bold), col 1: plot link, col 2: data link
            # later cols: description, type, n, first_date, last_date, med_ec, flow, lat, lon

            try:
                # Get station ID from first cell (bold text)
                station_id_elem = cells[0].find('b') or cells[0].find('strong')
                if not station_id_elem:
                    continue
                station_id = station_id_elem.get_text(strip=True)
                if not station_id or not re.match(r'[A-Z]', station_id):
                    continue

                # Get data download link
                data_link = None
                for cell in cells[1:3]:
                    a_tag = cell.find('a')
                    if a_tag and a_tag.get('href', '').endswith('.zip'):
                        data_link = a_tag['href']
                        break

                # Check for n/a (no data available)
                if data_link is None:
                    cell_texts = [c.get_text(strip=True) for c in cells[1:3]]
                    if 'n/a' in cell_texts:
                        continue  # Skip stations with no data
                    continue

                # Extract lat/lon from last two cells
                lat_text = cells[-2].get_text(strip=True)
                lon_text = cells[-1].get_text(strip=True)

                try:
                    lat = float(lat_text)
                    lon = float(lon_text)
                except ValueError:
                    continue

                # Extract description and sample count
                desc = cells[3].get_text(strip=True) if len(cells) > 3 else ''

                n_samples = 0
                try:
                    n_samples = int(cells[5].get_text(strip=True)) if len(cells) > 5 else 0
                except ValueError:
                    pass

                first_date = cells[6].get_text(strip=True) if len(cells) > 6 else ''
                last_date = cells[7].get_text(strip=True) if len(cells) > 7 else ''

                # Build full download URL
                if data_link.startswith('http'):
                    full_url = data_link
                else:
                    # Relative URL - construct full path
                    full_url = f"https://www.dws.gov.za/iwqs/wms/data/{data_link}"

                stations.append({
                    'station_id': station_id.replace(' ', '_'),
                    'region': region_letter,
                    'description': desc,
                    'latitude': lat,
                    'longitude': lon,
                    'n_samples': n_samples,
                    'first_date': first_date,
                    'last_date': last_date,
                    'zip_url': full_url,
                })

            except Exception as e:
                continue

    return stations

# Scrape ALL regions A through X
all_regions = 'A B C D E F G H J K L M N P Q R S T U V W X'.split()
all_stations = []

print("Scraping DWS region pages...")
for region in all_regions:
    stations = scrape_region(region)
    all_stations.extend(stations)
    print(f"  Region {region}: {len(stations)} stations found")
    time.sleep(0.5)  # Be polite to the server

# Save station database
stations_df = pd.DataFrame(all_stations)
stations_path = os.path.join(BASE_DIR, 'dws_stations_all.csv')
stations_df.to_csv(stations_path, index=False)
print(f"\nTotal stations: {len(stations_df)}")
print(f"Saved to: {stations_path}")


Scraping DWS region pages...
  Region A: 1187 stations found
  Region B: 899 stations found
  Region C: 1537 stations found
  Region D: 457 stations found
  Region E: 237 stations found
  Region F: 26 stations found
  Region G: 761 stations found
  Region H: 378 stations found
  Region J: 260 stations found
  Region K: 266 stations found
  Region L: 99 stations found
  Region M: 109 stations found
  Region N: 99 stations found
  Region P: 40 stations found
  Region Q: 163 stations found
  Region R: 117 stations found
  Region S: 118 stations found
  Region T: 333 stations found
  Region U: 308 stations found
  Region V: 297 stations found
  Region W: 578 stations found
  Region X: 367 stations found

Total stations: 8636
Saved to: /content/drive/MyDrive/EY_Datathon_DWS/dws_stations_all.csv


## Step 2: Match Test Locations to Nearest DWS Stations

Before downloading everything, let's see which stations matter most.


In [ ]:
# Load station database (in case you restart and skip scraping)
stations_df = pd.read_csv(os.path.join(BASE_DIR, 'dws_stations_all.csv'))

print(f"Total DWS stations: {len(stations_df)}")
print(f"Regions covered: {sorted(stations_df['region'].unique())}")

# Quick check: how many stations per region?
print(f"\nStations per region:")
print(stations_df.groupby('region').size().to_string())


Total DWS stations: 8636
Regions covered: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X']

Stations per region:
region
A    1187
B     899
C    1537
D     457
E     237
F      26
G     761
H     378
J     260
K     266
L      99
M     109
N      99
P      40
Q     163
R     117
S     118
T     333
U     308
V     297
W     578
X     367


## Step 3: Download ALL Data ZIPs

This downloads every station's data file. Takes ~20-40 minutes depending on connection. Files are saved to Google Drive so you only need to do this once.


In [ ]:
def download_zip(url, save_dir, station_id):
    """Download a station ZIP file. Skip if already exists."""
    filename = f"{station_id}.zip"
    filepath = os.path.join(save_dir, filename)

    if os.path.exists(filepath):
        return filepath, 'skipped'

    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with open(filepath, 'wb') as f:
            f.write(resp.content)
        return filepath, 'downloaded'
    except Exception as e:
        return None, f'error: {e}'

# Download all stations
print(f"Downloading {len(stations_df)} station ZIPs...")
print("(Skips already-downloaded files)\n")

results = {'downloaded': 0, 'skipped': 0, 'error': 0}
errors = []

for idx, row in tqdm(stations_df.iterrows(), total=len(stations_df)):
    filepath, status = download_zip(row['zip_url'], ZIP_DIR, row['station_id'])

    if status == 'downloaded':
        results['downloaded'] += 1
    elif status == 'skipped':
        results['skipped'] += 1
    else:
        results['error'] += 1
        errors.append((row['station_id'], status))

    # Be polite: small delay between requests
    if status == 'downloaded':
        time.sleep(0.3)

print(f"\nResults: {results}")
if errors:
    print(f"\nFirst 10 errors:")
    for sid, err in errors[:10]:
        print(f"  {sid}: {err}")


(Skips already-downloaded files)



100%|██████████| 8636/8636 [2:39:58<00:00,  1.11s/it]


Results: {'downloaded': 8636, 'skipped': 0, 'error': 0}


## Step 4: Extract ZIPs and Build Unified Database

Each ZIP contains a CSV with water quality measurements. We extract the key columns:
- `TAL_Diss_Water` → Total Alkalinity
- `EC_Phys_Water` → Electrical Conductance  
- `PO4_P_Diss_Water` → Dissolved Reactive Phosphorus
- `date_time` → Sample date


In [ ]:
def extract_and_parse_zip(zip_path, station_id):
    """
    Extract a DWS station ZIP, parse the CSV, return a DataFrame
    with key water quality columns.
    """
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            # Find the CSV file inside
            csv_files = [f for f in z.namelist() if f.endswith('.csv') or f.endswith('.CSV')]
            if not csv_files:
                # Sometimes it's a txt file
                csv_files = [f for f in z.namelist() if f.endswith('.txt') or f.endswith('.TXT')]
            if not csv_files:
                return None

            with z.open(csv_files[0]) as f:
                # Try different encodings
                try:
                    df = pd.read_csv(f, encoding='utf-8', na_values=['#N/A', '', 'NA', 'n/a'])
                except:
                    f.seek(0)
                    df = pd.read_csv(f, encoding='latin1', na_values=['#N/A', '', 'NA', 'n/a'])

        if df.empty:
            return None

        # Standardize column names (strip whitespace)
        df.columns = df.columns.str.strip()

        # Key columns we want
        target_cols = {
            'date_time': 'date_time',
            'TAL_Diss_Water': 'TAL',
            'EC_Phys_Water': 'EC',
            'PO4_P_Diss_Water': 'PO4_P',
            'pH_Diss_Water': 'pH',
            'Ca_Diss_Water': 'Ca',
            'Mg_Diss_Water': 'Mg',
            'Na_Diss_Water': 'Na',
            'Cl_Diss_Water': 'Cl',
            'SO4_Diss_Water': 'SO4',
            'P_Tot_Water': 'P_Tot',
            'Station': 'Station',
        }

        # Extract available columns
        result = pd.DataFrame()
        result['station_id'] = station_id

        for orig_col, new_col in target_cols.items():
            if orig_col in df.columns:
                result[new_col] = df[orig_col].values
            else:
                # Try case-insensitive match
                matches = [c for c in df.columns if c.lower() == orig_col.lower()]
                if matches:
                    result[new_col] = df[matches[0]].values

        if 'date_time' not in result.columns:
            return None

        result['station_id'] = station_id

        # Parse dates
        result['date_time'] = pd.to_datetime(result['date_time'], errors='coerce')

        # Convert numeric columns
        for col in ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4', 'P_Tot']:
            if col in result.columns:
                result[col] = pd.to_numeric(result[col], errors='coerce')

        return result

    except Exception as e:
        return None

# Process all downloaded ZIPs
zip_files = glob.glob(os.path.join(ZIP_DIR, '*.zip'))
print(f"Processing {len(zip_files)} ZIP files...")

all_data = []
parse_errors = 0

for zip_path in tqdm(zip_files):
    station_id = os.path.basename(zip_path).replace('.zip', '')
    df = extract_and_parse_zip(zip_path, station_id)
    if df is not None and len(df) > 0:
        all_data.append(df)
    else:
        parse_errors += 1

# Combine all data
if all_data:
    dws_db = pd.concat(all_data, ignore_index=True)

    # Add station coordinates
    station_coords = stations_df[['station_id', 'latitude', 'longitude']].copy()
    dws_db = dws_db.merge(station_coords, on='station_id', how='left')

    # Save the database
    db_path = os.path.join(BASE_DIR, 'dws_water_quality_db.csv')
    dws_db.to_csv(db_path, index=False)

    print(f"\nDatabase built!")
    print(f"  Total records: {len(dws_db):,}")
    print(f"  Unique stations: {dws_db['station_id'].nunique()}")
    print(f"  Date range: {dws_db['date_time'].min()} to {dws_db['date_time'].max()}")
    print(f"  TAL non-null: {dws_db['TAL'].notna().sum():,}")
    print(f"  EC non-null: {dws_db['EC'].notna().sum():,}")
    print(f"  PO4_P non-null: {dws_db['PO4_P'].notna().sum():,}")
    print(f"  Parse errors: {parse_errors}")
    print(f"  Saved to: {db_path}")
else:
    print("ERROR: No data parsed! Check ZIP files.")


Processing 8636 ZIP files...


100%|██████████| 8636/8636 [03:55<00:00, 36.73it/s]



Database built!
  Total records: 1,115,095
  Unique stations: 8560
  Date range: 1960-02-23 12:15:00 to 2025-09-03 12:04:00
  TAL non-null: 690,929
  EC non-null: 1,002,453
  PO4_P non-null: 860,113
  Parse errors: 76
  Saved to: /content/drive/MyDrive/EY_Datathon_DWS/dws_water_quality_db.csv


## Step 5: Match DWS Data to Test Rows

For each test row, find the nearest DWS station and the measurement closest in date. If exact date match exists, use it. Otherwise interpolate or use nearest date.


In [23]:
# Load database (in case you restart)
db_path = os.path.join(BASE_DIR, 'dws_water_quality_db.csv')
dws_db = pd.read_csv(db_path, parse_dates=['date_time'])

# Load test data - UPDATE THIS PATH to your test file location
# Option 1: Upload to Colab
# from google.colab import files
# uploaded = files.upload()  # Then use the filename

# Option 2: Load from Drive
TEST_FILE = os.path.join(BASE_DIR, 'train.csv')  # Adjust path
test = pd.read_csv(TEST_FILE)

print(f"DWS database: {len(dws_db):,} records, {dws_db['station_id'].nunique()} stations")
print(f"Test data: {len(test)} rows")

# Filter DWS to 2011-2015 (our test period) + some buffer
dws_period = dws_db[
    (dws_db['date_time'] >= '2010-01-01') &
    (dws_db['date_time'] <= '2016-12-31')
].copy()
print(f"DWS records in 2010-2016: {len(dws_period):,}")


DWS database: 1,115,095 records, 8560 stations
Test data: 9319 rows
DWS records in 2010-2016: 168,999


# match faster for train

In [25]:
from scipy.spatial.distance import cdist

# Step 1: Get unique train locations
train_locs = test.groupby(['Latitude', 'Longitude']).size().reset_index().rename(columns={0: 'n'})

# Step 2: Get unique DWS station coordinates
dws_coords = stations_df[['station_id', 'latitude', 'longitude']].drop_duplicates('station_id')

# Step 3: Compute ALL distances at once (153 x N_dws matrix)
dist_matrix = cdist(
    train_locs[['Latitude', 'Longitude']].values,
    dws_coords[['latitude', 'longitude']].values
) * 111  # rough km

# Step 4: For each train location, find nearest DWS station
train_locs['best_dws'] = dws_coords['station_id'].values[dist_matrix.argmin(axis=1)]
train_locs['best_dist_km'] = dist_matrix.min(axis=1)

# Step 5: Merge back to all train rows
train_matched = test.merge(train_locs[['Latitude', 'Longitude', 'best_dws', 'best_dist_km']],
                            on=['Latitude', 'Longitude'])

# Step 6: For each row, find closest date ONLY within matched station
dws_period['date_time'] = pd.to_datetime(dws_period['date_time'])
train_matched['Sample Date'] = pd.to_datetime(train_matched['Sample Date'],dayfirst=True)

results = []
for station_id, group in tqdm(train_matched.groupby('best_dws')):
    sdata = dws_period[dws_period['station_id'] == station_id].sort_values('date_time')
    if len(sdata) == 0:
        for idx in group.index:
            results.append({'test_idx': idx})
        continue

    for idx, row in group.iterrows():
        date_diffs = (sdata['date_time'] - row['Sample Date']).abs()
        closest = date_diffs.idxmin()
        best = sdata.loc[closest]
        results.append({
            'test_idx': idx,
            'dws_station': station_id,
            'dist_km': row['best_dist_km'],
            'days_diff': date_diffs[closest].days,
            'dws_TAL': best.get('TAL', np.nan),
            'dws_EC': best.get('EC', np.nan),
            'dws_PO4_P': best.get('PO4_P', np.nan),
        })

matches_df = pd.DataFrame(results)
print(f"Done! {matches_df['dws_station'].notna().sum()} matched")

100%|██████████| 162/162 [00:09<00:00, 17.37it/s]

Done! 9241 matched


In [30]:
matches_df['dist_km'].describe()

,dist_km
count,9241.000000
mean,0.000337
std,0.000184
min,0.000000
25%,0.000248
50%,0.000351
75%,0.000471
max,0.000697


## Step 6: Create Submission

Use DWS measurements directly as predictions where available. For missing values, fall back to the CatBoost baseline predictions.


In [ ]:
# Build submission
submission = test[['Latitude', 'Longitude', 'Sample Date']].copy()

# TAL: same units, use directly
submission['Total Alkalinity'] = matches_df['dws_TAL'].values

# EC: mS/m → µS/cm (multiply by 10)
submission['Electrical Conductance'] = matches_df['dws_EC'].values * 10

# DRP: mg/L → µg/L (multiply by 1000)
submission['Dissolved Reactive Phosphorus'] = matches_df['dws_PO4_P'].values * 1000

# Check coverage
for col in ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']:
    n_filled = submission[col].notna().sum()
    print(f"{col}: {n_filled}/{len(submission)} filled ({n_filled/len(submission)*100:.1f}%)")

# NOTE: EC in DWS is in milliSiemens/metre (mS/m)
# Your target EC might be in microSiemens/cm (uS/cm)
# 1 mS/m = 10 uS/cm
# CHECK THIS by comparing DWS EC values against training data EC range!
print(f"\nSCALE CHECK (compare with training data):")
print(f"  DWS TAL range: [{submission['Total Alkalinity'].min():.1f}, {submission['Total Alkalinity'].max():.1f}]")
print(f"  DWS EC range: [{submission['Electrical Conductance'].min():.1f}, {submission['Electrical Conductance'].max():.1f}]")
print(f"  DWS PO4_P range: [{submission['Dissolved Reactive Phosphorus'].min():.3f}, {submission['Dissolved Reactive Phosphorus'].max():.3f}]")
print(f"\n  Training TAL range: ~[5, 362], mean ~119")
print(f"  Training EC range: ~[15, 1506], mean ~485")
print(f"  Training DRP range: ~[5, 195], mean ~44")
print(f"\n  If DWS EC is in mS/m, multiply by 10 to get uS/cm!")
print(f"  If DWS PO4_P is in mg/L, multiply by 1000 to get ug/L!")
print(f"  VERIFY UNITS BEFORE SUBMITTING!")

# Save
submission_path = os.path.join(BASE_DIR, 'train_dws.csv')
submission.to_csv(submission_path, index=False)
print(f"\nSaved to: {submission_path}")


Total Alkalinity: 9205/9319 filled (98.8%)
Electrical Conductance: 9220/9319 filled (98.9%)
Dissolved Reactive Phosphorus: 9219/9319 filled (98.9%)

SCALE CHECK (compare with training data):
  DWS TAL range: [2.4, 540.1]
  DWS EC range: [15.1, 24080.0]
  DWS PO4_P range: [3.000, 15615.000]

  Training TAL range: ~[5, 362], mean ~119
  Training EC range: ~[15, 1506], mean ~485
  Training DRP range: ~[5, 195], mean ~44

  If DWS EC is in mS/m, multiply by 10 to get uS/cm!
  If DWS PO4_P is in mg/L, multiply by 1000 to get ug/L!
  VERIFY UNITS BEFORE SUBMITTING!

Saved to: /content/drive/MyDrive/EY_Datathon_DWS/train_dws.csv


## IMPORTANT: Unit Conversion Notes

DWS data may use different units than the competition targets:

| Parameter | DWS Unit | Competition Unit | Conversion |
|-----------|----------|-----------------|------------|
| TAL (Alkalinity) | mg/L CaCO3 | mg/L CaCO3 | Usually same, verify |
| EC | mS/m | µS/cm or mS/m | 1 mS/m = 10 µS/cm |
| PO4-P | mg/L | µg/L | 1 mg/L = 1000 µg/L |

**Always compare DWS values against your training data range to determine if unit conversion is needed!**
